# Recherches après 1365.745

Le meilleur score Kaggle transmis par l'utilisateur est 1365.745. Les cellules lisent les résultats des recherches locales. Le modèle original et toutes les soumissions précédentes sont conservés. Les clients et le leaderboard ont déjà été étudiés : les résultats restent diagnostiques.

In [1]:
from pathlib import Path
import json
import pandas as pd
ROOT=next(p for p in [Path.cwd(),Path.cwd().parent] if (p/'run.py').exists())
names=['smooth_severity','coverage_interactions','frequency_curves']
results={n:json.loads((ROOT/'reports'/n/'results.json').read_text()) for n in names}
pd.DataFrame([{'search':n,'RMSE':r['selected']['RMSE'],'incumbent_RMSE':r['incumbent']['RMSE'],'gain':r['gain']['gain_RMSE'],'export':r['export']} for n,r in results.items()])

,search,RMSE,incumbent_RMSE,gain,export
0,smooth_severity,542.000491,541.971031,-0.02946,None
1,coverage_interactions,541.986791,541.971031,-0.01576,None
2,frequency_curves,541.915571,541.971031,0.05546,{'file': 'deliverables\candidates\submission_f...


La recette fixe du modèle à 1365.745 est réapprise par pli. Sa RMSE de 541.971031 diffère de la RMSE 541.942903 de la précédente procédure, qui sélectionnait des configurations différentes selon les plis. Les courbes de gravité affinées et les interactions couverture/usage dégradent cette référence fixe et n'exportent aucun CSV.

La recherche de fréquence apprend une correction de logit des probabilités originales, avec splines et pénalisation L2 forte. Les probabilités auxiliaires sont calculées hors client dans trois sous-plis supplémentaires ; aucune probabilité ajustée sur son propre client n'est utilisée pour apprendre la correction. Pas de poids de classe ni de repondération des observations. La gravité du modèle à 1365.745 reste exactement inchangée. Trois plis externes évaluent toute la sélection dans cinq plis internes.

In [2]:
r=results['frequency_curves']
pd.DataFrame([{'fold':x['fold'],'RMSE':x['selected']['RMSE'],'incumbent_RMSE':x['incumbent']['RMSE'],'choice':x['choice']} for x in r['folds']])

,fold,RMSE,incumbent_RMSE,choice
0,1,569.403510,569.479756,"{'candidate': 3, 'weight': 1.0, 'selection_RMS..."
1,2,517.966777,518.038670,"{'candidate': 3, 'weight': 1.0, 'selection_RMS..."
2,3,537.128670,537.146373,"{'candidate': 3, 'weight': 1.0, 'selection_RMS..."


In [3]:
r['gain'],r['frequency']

({'gain_RMSE': 0.05546031776543714,
  'interval_95': [0.0046495906212925325, 0.11078638878312856],
  'repetitions': 2000,
  'interpretation': 'Descriptive paired client bootstrap conditional on fitted predictions; not full training uncertainty.'},
 {'selected': {'log_loss': 0.2139301673711023, 'brier': 0.054071440662871346},
  'incumbent': {'log_loss': 0.21404432055967237,
   'brier': 0.054085306146468504}})

In [4]:
pd.read_csv(ROOT/'reports/frequency_curves/contract_performance.csv')

,type_contrat,rows,selected_RMSE,incumbent_RMSE
0,Maxi,32360,626.926819,626.981661
1,Median1,4638,440.971996,440.964459
2,Median2,8759,291.071640,291.206752
3,Mini,4243,275.037699,275.090002


In [5]:
s=json.loads((ROOT/'reports/frequency_curves/full_training/selection.json').read_text())
print('Poids :',r['final_choice']['weight'])
s['catalogue'][r['final_choice']['candidate']] if r['final_choice']['candidate']>=0 else 'Incumbent inchangé'

Poids : 1.0


{'panel': 'age_coverage', 'alpha': 100.0}

In [6]:
verification=json.loads((ROOT/'reports/frequency_curves/verification.json').read_text())
assert verification['status']=='verified'
verification

{'status': 'verified',
 'tests_passed': 57,
 'original_notebook_sha256': '76248bdb51400fe399c670a9b71092cabe041cc13d3839b83846ac1499938429',
 'incumbent_csv_unchanged': True,
 'best_user_reported_public_score': 1365.657,
 'source_hashes': {'refine_smooth_severity.py': 'c0279d0ba901856abbb6efa9ae9bfd21175e7e7b152764b6d6e211cc29266892',
  'coverage_interactions.py': '827d50b1531d886c6ec94c69c8abbf5629d56c18a8794eb5c473afd83afcd06c',
  'frequency_curves.py': '379ad2938df47369477d44018ced32b4f1641f71f2a3b77efbf6e4afd56bd7c1'},
 'provenance': {'data': {'train.csv': '8b5ca8a59c214269056ed1720de33619c10968549c383ec350898a659b97bfc5',
   'test.csv': '68cdf87201974688d754d846c2d8ff33c5aa2f9c011e84fba92018186cd8f569'},
  'source': {'__init__.py': '100b3e376bb5c6d141565df62567e20d231861a23f17cb8e15d587cce02ee61e',
   'experiment.py': '0237a7515b456a562c85c8b5e7339f2d08ca5a179c75bb7c7da33f6c86522eab',
   'models.py': 'a2787421cff97fa7ca2f7fb21d863eec035c61cd6826f459bbcb476d0aad38a8',
   'reference

La sélection minimise la RMSE de prime. Les intervalles bootstrap clients sont descriptifs, conditionnels aux prédictions ajustées. Une amélioration locale ne garantit pas une amélioration Kaggle ; aucun nouveau score public n'est inventé. Les exports, lorsqu'ils existent, sont reproduits octet pour octet dans un processus distinct. Commandes et résultats : `docs/variable_followups.md`.